In [ ]:
import pandas as pd
from google.oauth2 import service_account
from googleapiclient.discovery import build
from datetime import date
from pathlib import Path

# --- Google Sheet details ---
SPREADSHEET_ID = "11L6GRPLvBqZU0TxuYaUuSH8T74elONg_qKWASThF7vI"
SHEET_NAME = "training set"   # exact tab name
RANGE_NAME = f"'{SHEET_NAME}'!B:HW"  # columns B–HW only

# --- Authenticate with Service Account ---
creds = service_account.Credentials.from_service_account_file(
    "service_account.json",
    scopes=["https://www.googleapis.com/auth/spreadsheets.readonly"]
)

# --- Connect to Sheets API ---
service = build("sheets", "v4", credentials=creds)
sheet = service.spreadsheets()
result = sheet.values().get(spreadsheetId=SPREADSHEET_ID, range=RANGE_NAME).execute()
values = result.get("values", [])

# --- Convert to DataFrame + Save ---
if values:
    df = pd.DataFrame(values[1:], columns=values[0])

    # save file under your repo workspace
    out_dir = Path("/workspaces/NBA-model/data")
    out_dir.mkdir(parents=True, exist_ok=True)
    filename = out_dir / "training_set.csv"

    df.to_csv(filename, index=False)
    print(f"✅ Saved latest training set ({len(df)} rows) to {filename}")
else:
    print("⚠️ No data found in the specified range.")
